# S14 - Híbridos y guardarraíles (Pagarium) — Soluciones

Cuaderno **de soluciones** de la S14 (RA5-b/d/e). Se corresponde con `sesion14_hibridos_guardarrailes.ipynb`.

## 1. FIGS redescubre la política latente

In [1]:
import numpy as np
from imodels import FIGSClassifier
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
n = 4000
importe = rng.uniform(0, 2000, n)
antiguedad = rng.uniform(0, 24, n)
n_intentos = rng.integers(0, 6, n)

y = ((importe > 1000) & (antiguedad < 6)) | (n_intentos >= 4)
flip = rng.random(n) < 0.03
y = np.where(flip, ~y, y).astype(int)

X = np.column_stack([importe, antiguedad, n_intentos])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

clf = FIGSClassifier(max_rules=6)
clf.fit(X_tr, y_tr)
print("Accuracy:", round(clf.score(X_te, y_te), 3))
print(clf)

Accuracy: 0.972
FIGSClassifier
  FIGS: a sum of small trees.
  depth 3  ·  7 leaves  ·  2,800 training samples  ·  classes: 0, 1
  Each tree adds the value of the leaf a sample reaches; P(1) = sigmoid(sum).

  X2 ≤ 3.5 ?               2,800 samples
  ├─ yes X1 ≤ 6.003 ?      1,882
  │  ├─ yes X0 ≤ 1,002 ?     496
  │  │  ├─ yes → -0.9137     255
  │  │  └─ no  → +0.9336     241
  │  └─ no  X0 ≤ 1,907 ?   1,386
  │     ├─ yes → -0.95     1,319
  │     └─ no  → -0.8507      67
  └─ no  X0 ≤ 1,991 ?        918
     ├─ yes → +0.9278        914
     └─ no  X0 ≤ 1,993 ?       4
        ├─ yes → -1            1
        └─ no  → +1            3


FIGS recupera reglas legibles: `X0 ≈ importe`, `X1 ≈ antiguedad`, `X2 ≈ n_intentos`. Observa cómo aparecen umbrales cercanos a los reales (`importe > 1000`, `antiguedad < 6`, `n_intentos >= 4`), con márgenes por el ruido.

## 2. Guardarraíl de un agente LLM (base)

In [2]:
import re

def validar_accion_llm(texto):
    importes = [float(x) for x in re.findall(r"(\d+(?:\.\d+)?)\s*EUR", texto)]
    if any(i > 1000 for i in importes):
        return False, "importe fuera de rango"
    if "cuenta no verificada" in texto.lower():
        return False, "destino no permitido"
    if re.search(r"transferir.*sin confirmar", texto, re.IGNORECASE):
        return False, "requiere confirmacion humana"
    return True, "ok"

for t in ["Transferir 500 EUR a la cuenta A",
          "Transferir 5000 EUR",
          "Pagar a cuenta no verificada"]:
    print(t, "->", validar_accion_llm(t))

Transferir 500 EUR a la cuenta A -> (True, 'ok')
Transferir 5000 EUR -> (False, 'importe fuera de rango')
Pagar a cuenta no verificada -> (False, 'destino no permitido')


## 3. Solución: las dos capas del proyecto Pagarium

- **Capa de negocio:** la tabla DMN de la S11, con `rule-engine`.
- **Capa guardarraíl:** `validar_accion_llm` + una **regla nueva** (bloquear divisas no permitidas).

In [3]:
import re, rule_engine

# ---- Capa 1: negocio (tabla DMN de la S11) ----
reglas = {
    "aprobar":          rule_engine.Rule("importe < 500 and antiguedad_meses >= 6 and n_intentos < 4"),
    "revisar_intentos": rule_engine.Rule("importe < 500 and antiguedad_meses >= 6 and n_intentos >= 4"),
    "revisar_importe":  rule_engine.Rule("importe >= 500"),
    "rechazar":         rule_engine.Rule("antiguedad_meses < 6"),
}
def decidir(c):
    return [n for n, r in reglas.items() if r.matches(c)]

# ---- Capa 2: guardarraíl (reglas sobre la salida del LLM) ----
def validar_accion_llm(texto):
    importes = [float(x) for x in re.findall(r"(\d+(?:\.\d+)?)\s*EUR", texto)]
    if any(i > 1000 for i in importes):
        return False, "importe fuera de rango"
    if "cuenta no verificada" in texto.lower():
        return False, "destino no permitido"
    if re.search(r"transferir.*sin confirmar", texto, re.IGNORECASE):
        return False, "requiere confirmacion humana"
    divisas = re.findall(r"\b(USD|GBP|JPY|CHF)\b", texto.upper())   # REGLA NUEVA
    if divisas:
        return False, f"divisa no permitida: {divisas[0]}"
    return True, "ok"

print("capa negocio:", decidir({"importe": 200, "antiguedad_meses": 10, "n_intentos": 2}))
print("solape:", decidir({"importe": 600, "antiguedad_meses": 3, "n_intentos": 2}))

for t in ["Transferir 500 EUR a la cuenta A",
          "Transferir 5000 EUR",
          "Pagar a cuenta no verificada",
          "Transferir 300 USD a la cuenta B"]:
    print(t, "->", validar_accion_llm(t))

capa negocio: ['aprobar']
solape: ['revisar_importe', 'rechazar']
Transferir 500 EUR a la cuenta A -> (True, 'ok')
Transferir 5000 EUR -> (False, 'importe fuera de rango')
Pagar a cuenta no verificada -> (False, 'destino no permitido')
Transferir 300 USD a la cuenta B -> (False, 'divisa no permitida: USD')


**Conclusión (3 líneas):** sí usaría un motor, pero **ligero** (p. ej. `rule-engine` o ZEN), porque las reglas de elegibilidad cambian a menudo y las gestiona negocio; para 4 reglas fijas bastaría un `if/else`. El **guardarraíl** es imprescindible: el LLM propone, pero no debe poder ejecutar fuera de los límites (importe, destino, divisa). El benchmark confirma que la capa de reglas es barata en valor y cara en microsegundos: se paga con la **trazabilidad** y el **cumplimiento (AI Act)** que aporta.